# A Comparative Study of Methods for Early Breast Cancer Detection Using Medical Data

**Models:** Decision Tree · Random Forest · SVM  
**Dataset:** Wisconsin Breast Cancer Dataset (WBCD)  
**Pipeline:** Data Acquisition → Preprocessing → Feature Selection → Training → Evaluation → Comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

RANDOM_STATE = 42
TEST_SIZE = 0.20
sns.set_theme(style="whitegrid")

## 1. Data Acquisition

In [ ]:
df = pd.read_csv("data.csv")
df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]
empty_cols = [c for c in df.columns if str(c).strip() == ""]
df = df.drop(columns=empty_cols, errors="ignore")
if "id" in df.columns:
    df = df.drop(columns=["id"])

print("Shape:", df.shape)
print(df["diagnosis"].value_counts())
df.head()

## 2. Data Preprocessing

- Encode labels: **Malignant = 1** (positive class), **Benign = 0**
- Remove highly correlated features
- 80/20 stratified train–test split
- StandardScaler normalization

In [ ]:
y = df["diagnosis"].map({"M": 1, "B": 0}).astype(int)
X = df.drop(columns=["diagnosis"])

print("Missing values:", int(X.isna().sum().sum()))
print("Benign:", int((y == 0).sum()), "| Malignant:", int((y == 1).sum()))

In [ ]:
# Correlation heatmap (optional visualization)
plt.figure(figsize=(12, 10))
sns.heatmap(X.corr(), cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": 0.6})
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
CORR_THRESHOLD = 0.92
corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]
X = X.drop(columns=to_drop)
print(f"Dropped {len(to_drop)} highly correlated features → {X.shape[1]} remain")
print("Dropped:", to_drop)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f"Train: {X_train_scaled.shape[0]} | Test: {X_test_scaled.shape[0]}")

## 3. Feature Selection (ANOVA F-test)

In [ ]:
N_FEATURES = 15
selector = SelectKBest(score_func=f_classif, k=min(N_FEATURES, X_train_scaled.shape[1]))
X_train_fs = selector.fit_transform(X_train_scaled, y_train)
X_test_fs = selector.transform(X_test_scaled)
selected = list(X_train_scaled.columns[selector.get_support()])

X_train_fs = pd.DataFrame(X_train_fs, columns=selected, index=X_train_scaled.index)
X_test_fs = pd.DataFrame(X_test_fs, columns=selected, index=X_test_scaled.index)
print("Selected features:", selected)

## 4–5. Model Training & Testing

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "SVM": SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE),
}

rows = []
predictions = {}

for name, model in models.items():
    model.fit(X_train_fs, y_train)
    y_pred = model.predict(X_test_fs)
    predictions[name] = y_pred
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred) * 100,
        "Precision": precision_score(y_test, y_pred, pos_label=1) * 100,
        "Recall": recall_score(y_test, y_pred, pos_label=1) * 100,
        "F1-Score": f1_score(y_test, y_pred, pos_label=1) * 100,
    })
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=["Benign", "Malignant"]))

## 6. Performance Evaluation & Comparative Analysis

In [ ]:
results = pd.DataFrame(rows)
results[["Accuracy", "Precision", "Recall", "F1-Score"]] = results[
    ["Accuracy", "Precision", "Recall", "F1-Score"]
].round(2)
results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, name in zip(axes, predictions):
    cm = confusion_matrix(y_test, predictions[name])
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", ax=ax,
        xticklabels=["Benign", "Malignant"],
        yticklabels=["Benign", "Malignant"],
    )
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
plot_df = results.melt(id_vars="Model", var_name="Metric", value_name="Score (%)")
plt.figure(figsize=(9, 5))
sns.barplot(data=plot_df, x="Metric", y="Score (%)", hue="Model")
plt.ylim(85, 100)
plt.title("Model Comparison — Breast Cancer Detection (WBCD)")
plt.legend(title="Model", loc="lower right")
plt.tight_layout()
plt.show()

best = results.loc[results["Accuracy"].idxmax()]
print(f"Best model: {best['Model']} (Accuracy={best['Accuracy']:.2f}%, Recall={best['Recall']:.2f}%)")

In [ ]:
# Feature importance (Random Forest)
rf = models["Random Forest"]
importance = pd.Series(rf.feature_importances_, index=selected).sort_values(ascending=False)
plt.figure(figsize=(8, 5))
importance.plot(kind="barh", color="#2a6f97")
plt.gca().invert_yaxis()
plt.title("Random Forest — Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()